In [1]:
%load_ext autoreload

In [ ]:
%autoreload 2
from utils import (
    select_users_by_period,
    create_hourly_user_dataset,
    compute_position_metrics,
)
from visualization_utils import (
    plot_market_features,
)


In [3]:
# =============================================================================
# IMPORTS
# =============================================================================
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore")
# =============================================================================
# HELPER FUNCTIONS (unchanged logic)
# =============================================================================
def get_users_with_leverage(df_actions, start_date, end_date, window_hours=24, threshold_date=None):
    filtered_actions = select_users_by_period(df_actions, start_date, end_date, threshold_date)
    
    open_events = filtered_actions[filtered_actions['event_sequence_type'] == 'position_open'][['user_address', 'timestamp']].copy()
    open_events = open_events.rename(columns={'timestamp': 'open_time'})
    
    leverage_counts = []
    for user in open_events['user_address'].unique():
        user_open = open_events[open_events['user_address'] == user]['open_time'].iloc[0]
        user_df = filtered_actions[filtered_actions['user_address'] == user]
        
        window_end = user_open + window_hours * 3600
        window_events = user_df[(user_df['timestamp'] >= user_open) & (user_df['timestamp'] <= window_end)]
        
        leverage_count = len(window_events[window_events['event_sequence_type'] == 'borrow_more_w_collateral']) // 2
        leverage_counts.append({'user_address': user, 'leverage_factor': leverage_count})
    
    leverage_df = pd.DataFrame(leverage_counts)
    result = filtered_actions.merge(leverage_df, on='user_address', how='left')
    result["leverage_factor"] += 1

    if threshold_date is not None:
        result = result[result["datetime"].astype(str) < str(threshold_date)]
    
    return result

def add_time_difference_features(hourly_df, lookback_hours=[3, 6], diff_fields=['ltv', 'collateral_price', 'market_utilization', 'borrow_rate']):
    df = hourly_df.sort_values(['user_address', 'timestamp']).copy()
    
    open_times = df.groupby('user_address')['timestamp'].min().reset_index()
    open_times.columns = ['user_address', 'open_timestamp']
    df = df.merge(open_times, on='user_address', how='left')
    
    open_values = df[df['timestamp'] == df['open_timestamp']].copy()
    open_values_dict = {}
    for field in diff_fields:
        open_values_dict[f'open_{field}'] = open_values.set_index('user_address')[field].to_dict()
    
    for field in diff_fields:
        df[f'{field}_vs_open'] = df.apply(
            lambda row: row[field] - open_values_dict[f'open_{field}'].get(row['user_address'], 0), 
            axis=1
        )
    
    for hours in lookback_hours:
        for field in diff_fields:
            df[f'{field}_{hours}h_ago'] = df.groupby('user_address')[field].shift(hours)
            df[f'{field}_vs_{hours}h'] = df[field] - df[f'{field}_{hours}h_ago']
            df = df.drop(columns=[f'{field}_{hours}h_ago'])
    
    df = df.drop(columns=['open_timestamp'])
    return df.fillna(0)

leave_reasons = []
def remove_similar_hours(hourly_df, diff_thresholds, min_events_cnt=5):
    global leave_reasons
    leave_reasons = []
    df = hourly_df.sort_values(['user_address', 'timestamp']).copy()
    result_dfs = []
    
    for user, user_df in df.groupby('user_address'):
        action_mask = user_df['action'] != 'none'
        keep_indices = set(user_df[action_mask].index)
        
        last_kept_idx = None
        
        for idx, curr_row in user_df.iterrows():
            if idx in keep_indices or last_kept_idx is None:
                keep_indices.add(idx)
                last_kept_idx = idx
                continue
            
            prev_row = user_df.loc[last_kept_idx]
            similar = True
            for col, threshold in diff_thresholds.items():
                if col in curr_row and col in prev_row:
                    diff = abs(curr_row[col] - prev_row[col])
                    if diff > threshold:
                        leave_reasons.append(col)
                        similar = False
                        break
            if not similar:
                keep_indices.add(idx)
                last_kept_idx = idx
        
        if len(keep_indices) < min_events_cnt:
            keep_indices = set(user_df[action_mask].index)
        
        if keep_indices:
            result_dfs.append(user_df.loc[list(keep_indices)])
    
    return pd.concat(result_dfs).sort_values(['user_address', 'timestamp'])

def create_model_dataset(hourly_user_df, target_horizon_hours=1, leverage_factor_min=1, liq_threshold=0.86):
    df = hourly_user_df.sort_values(['user_address', 'timestamp']).copy()
    
    if leverage_factor_min > 0:
        df = df[df["leverage_factor"] >= leverage_factor_min]
    
    df["log_debt"] = np.log1p(df["debt"])
    df['has_action'] = (df['action'] != 'none').astype(int)
    
    open_times = df.groupby('user_address')['timestamp'].min().rename('open_timestamp')
    df = df.join(open_times, on='user_address')
    df['days_since_open'] = (df['timestamp'] - df['open_timestamp']) / 3600.0 / 24.0
    
    def compute_hours_since_last_action(group):
        group = group.sort_values('timestamp')
        last_action_ts = None
        res = []
        for ts, action in zip(group['timestamp'], group['has_action']):
            if action == 1:
                res.append(0.0)
                last_action_ts = ts
            else:
                if last_action_ts is None:
                    res.append(1000.0)
                else:
                    res.append((ts - last_action_ts) / 3600.0)
        return pd.Series(res, index=group.index)
    
    df['days_since_last_action'] = df.groupby('user_address', group_keys=False).apply(compute_hours_since_last_action) // 12
    
    def action_count_last_n_hours(group, window_hours=6):
        group = group.sort_values('timestamp')
        timestamps = group['timestamp'].values
        actions = group['has_action'].values
        counts = []
        for i, ts in enumerate(timestamps):
            start = ts - window_hours * 3600
            left = np.searchsorted(timestamps, start, side='left')
            count = np.sum(actions[left:i])
            counts.append(count)
        return pd.Series(counts, index=group.index)
    
    df['action_count_last_6h'] = df.groupby('user_address', group_keys=False).apply(
        lambda g: action_count_last_n_hours(g, 6)
    )
    
    df['ltv_change_1h'] = df.groupby('user_address')['ltv'].diff(1)
    df['distance_to_liq'] = np.maximum(liq_threshold - df['ltv'], 0)
    
    if 'borrow_rate_rolling' in df.columns:
        df['borrow_rate_trend'] = df['borrow_rate'] - df['borrow_rate_rolling']
    else:
        df['borrow_rate_trend'] = df.groupby('user_address')['borrow_rate'].transform(
            lambda x: x - x.rolling(6, min_periods=1).mean()
        )
    
    df['ltv_times_rate'] = df['ltv'] * df['borrow_rate']
    df['debt_change_1h'] = df.groupby('user_address')['debt'].diff(1)
    df['volatility_ltv'] = df['volatility_6h'] * df['ltv']
    df['hour_of_day'] = pd.to_datetime(df['timestamp'], unit='s').dt.hour
    df['day_of_week'] = pd.to_datetime(df['timestamp'], unit='s').dt.dayofweek
    
    def has_action_in_future(group):
        results = []
        timestamps = group['timestamp'].values
        actions = group['has_action'].values
        for i in range(len(group) - 1):
            current_ts = timestamps[i]
            horizon_ts = current_ts + target_horizon_hours * 3600
            future_indices = [j for j in range(i+1, len(group)) if timestamps[j] <= horizon_ts]
            results.append(1 if future_indices and any(actions[future_indices]) else 0)
        results.append(0)
        return pd.Series(results, index=group.index)
    
    df['action_next'] = df.groupby('user_address', group_keys=False).apply(has_action_in_future)
    
    features = [
        "borrow_rate_vs_open",
        "ltv_vs_open",
        "collateral_price_vs_open",
        "collateral_price_vs_6h",
        "volatility_6h",
        "drawdown_6h",
        "hours_since_open",
        "hours_since_last_action",
        "action_count_last_6h",
        "ltv_change_1h",
        "distance_to_liq",
        "borrow_rate_trend",
        "ltv_times_rate",
        "debt_change_1h",
        "volatility_ltv",
        "hour_of_day",
        "day_of_week",
        "log_debt",
    ]
    features = [f for f in features if f in df.columns]
    
    return df[['user_address', 'timestamp'] + features + ['action_next']].fillna(0)

def prepare_features(df):
    df = df.copy()
    categorical_cols = ['hour_of_day', 'day_of_week']
    feature_cols = [col for col in df.columns if col not in ['user_address', 'timestamp', 'action_next']]
    
    for col in categorical_cols:
        if col in feature_cols:
            feature_cols.remove(col)
    
    if 'ltv' in df.columns:
        df['ltv_squared'] = df['ltv'] ** 2
        feature_cols.append('ltv_squared')
    
    if 'ltv' in df.columns and 'hours_since_open' in df.columns:
        df['ltv_time'] = df['ltv'] * df['hours_since_open']
        feature_cols.append('ltv_time')
    
    if 'distance_to_liq' in df.columns:
        df['log_dist_to_liq'] = np.log1p(df['distance_to_liq'])
        feature_cols.append('log_dist_to_liq')
    
    if 'ltv_times_rate' not in feature_cols and 'ltv' in df.columns and 'borrow_rate' in df.columns:
        df['ltv_times_rate'] = df['ltv'] * df['borrow_rate']
        feature_cols.append('ltv_times_rate')
    
    dummies = pd.get_dummies(df[categorical_cols], drop_first=True)
    dummy_names = dummies.columns.tolist()
    
    X = pd.concat([df[feature_cols], dummies], axis=1)
    y = df['action_next'].copy()
    
    feature_names = feature_cols + dummy_names
    return X, y, feature_names, dummy_names

def train_logistic_model_statsmodels(model_df, verbose=True):
    X, y, feature_names, dummy_columns = prepare_features(model_df)
    
    split_idx = int(len(X) * 0.8)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
    
    X_train = X_train.replace([np.inf, -np.inf], np.nan)
    X_test = X_test.replace([np.inf, -np.inf], np.nan)
    
    for col in X_train.columns:
        median_val = X_train[col].median()
        X_train[col].fillna(median_val, inplace=True)
        X_test[col].fillna(median_val, inplace=True)
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    X_train_sm = sm.add_constant(X_train_scaled)
    X_test_sm = sm.add_constant(X_test_scaled)
    model_sm = sm.Logit(y_train, X_train_sm)
    result = model_sm.fit(disp=0, maxiter=1000)
    
    full_feature_names = ['const'] + feature_names
    result.params.index = full_feature_names
    result.bse.index = full_feature_names
    
    y_prob = result.predict(X_test_sm)
    roc_auc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)
    metrics = {'roc_auc': roc_auc, 'pr_auc': pr_auc}
    
    if verbose:
        print("\n=== Model Performance ===")
        print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
        print(f"Action rate in test: {y_test.mean():.3f}")
        print(f"ROC-AUC: {roc_auc:.3f}")
        print(f"PR-AUC: {pr_auc:.3f}")
        
        coef_df = pd.DataFrame({
            'feature': full_feature_names,
            'coefficient': result.params.values,
            'p_value': result.pvalues.values,
            'std_err': result.bse.values
        })
        coef_df['significant'] = coef_df['p_value'] < 0.05
        coef_df['sign'] = ['+' if c > 0 else '-' for c in coef_df['coefficient']]
        coef_df = coef_df.sort_values('p_value')
        
        print("\n=== Feature Significance (sorted by p-value) ===")
        print(coef_df[['feature', 'coefficient', 'p_value', 'significant', 'sign']].to_string(index=False))
        
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, label=f'ROC (AUC = {roc_auc:.3f})')
        plt.plot([0, 1], [0, 1], 'k--')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('ROC Curve')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()
    
    p_values = pd.Series(result.pvalues.values, index=full_feature_names)
    coefs = pd.Series(result.params.values, index=full_feature_names)
    return result, scaler, p_values, coefs, metrics

def compute_elasticity(model_df, result, scaler, feature_names, features_of_interest):
    X, y, _, _ = prepare_features(model_df)
    X = X[feature_names]
    
    X = X.replace([np.inf, -np.inf], np.nan)
    
    X_scaled = scaler.transform(X)
    X_sm = sm.add_constant(X_scaled)
    
    p = result.predict(X_sm)
    p = p[~np.isnan(p)]
    p_mean = p.mean()
    
    coef_values = result.params.values[1:]  # exclude const
    
    elasticities = {}
    for feat in features_of_interest:
        if feat not in feature_names:
            elasticities[feat] = np.nan
            continue
        
        idx = feature_names.index(feat)
        beta = coef_values[idx]
        x_mean = X[feat].mean()
        elasticity = beta * x_mean * (1 - p_mean)
        elasticities[feat] = elasticity
    
    return elasticities, p_mean

# if __name__ == "__main__":
#     main()

In [169]:
# =============================================================================
# MAIN PIPELINE
# =============================================================================
    # Parameters
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 200)


def load_data(market_name, events_path, hourly_path):
    """Load events and hourly market data for a given market."""
    df = pd.read_csv(f"{events_path}/{market_name}.csv")
    market_df = pd.read_csv(f"{hourly_path}/{market_name}.csv")
    return df, market_df

def add_yield_to_actions(actions_df, yield_df):
    df = actions_df.copy()
    yield_df = yield_df.copy()
    yield_df['timestamp'] = pd.to_datetime(yield_df['timestamp']).astype('int64') // 10**9
    
    # Get min and max timestamps from yield data
    yield_min = yield_df['timestamp'].min()
    yield_max = yield_df['timestamp'].max()
    
    # Filter actions
    df = df[(df['timestamp'] >= yield_min) & (df['timestamp'] <= yield_max + 30 * 24 * 60 * 60)]
    df = df.sort_values('timestamp')
    yield_df = yield_df.sort_values('timestamp')
    df = pd.merge_asof(
        df,
        yield_df[['timestamp', 'base_apy', 'implied_apy', 'underlying_apy']],
        on='timestamp',
        direction='nearest'
    )
    
    return df

TOKEN = "PT-USDe-25SEP2025"
MARKET = f"eth_{TOKEN}_usdc"
EVENTS_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_enriched"
HOURLY_MARKET_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_hourly_data"
PT_YIELDS_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/pt_yields"

# Load data
df, market_df = load_data(MARKET, EVENTS_PATH, HOURLY_MARKET_PATH)
yield_df = pd.read_csv(f"{PT_YIELDS_PATH}/{TOKEN}.csv").sort_values("timestamp")
df = add_yield_to_actions(df, yield_df)
market_df = add_yield_to_actions(market_df, yield_df)

In [44]:

# =========================================================================
# Create monthly cohorts
# =========================================================================

def create_cohort_intervals(df_actions, token_type=None, n_intervals=10):
    """
    Create cohort intervals based on token expiry if token_type is provided.
    token_type format: e.g., "PT-USDe-25SEP2025"
    """
    min_ts = df_actions['timestamp'].min()
    max_ts = df_actions['timestamp'].max()
    start_date = pd.to_datetime(min_ts, unit='s')
    end_date = pd.to_datetime(max_ts, unit='s')
    
    # If token_type provided, parse expiry and set as upper bound
    if token_type:
        import re
        match = re.search(r'(\d{2}[A-Z]{3}\d{4})', token_type)
        if match:
            expiry_str = match.group(1)
            expiry_date = pd.to_datetime(expiry_str, format='%d%b%Y')
            end_date = min(end_date, expiry_date)
    
    total_days = (end_date - start_date).days
    interval_days = total_days // n_intervals
    
    intervals = []
    for i in range(n_intervals):
        interval_start = start_date + pd.Timedelta(days=i * interval_days)
        interval_end = start_date + pd.Timedelta(days=(i+1) * interval_days)
        threshold = min(interval_end + pd.DateOffset(months=4), end_date + pd.DateOffset(months=1))
        
        intervals.append((interval_start, interval_end, threshold))
    
    return intervals

intervals = create_cohort_intervals(df, TOKEN, n_intervals=3)
monthly_dfs = []

for interval_start, interval_end, threshold in tqdm(intervals):
    start_ts = int(interval_start.timestamp())
    end_ts = int(interval_end.timestamp())
    
    user_df = get_users_with_leverage(df, interval_start, interval_end, threshold_date=threshold)
    monthly_dfs.append(user_df)

    print(f"Cohort {interval_start}-{interval_end}, threshold {threshold} - {len(user_df)} events")


  0%|          | 0/3 [00:00<?, ?it/s]

Cohort 2025-07-25 14:33:35-2025-08-14 14:33:35, threshold 2025-10-25 00:00:00 - 1762 events
Cohort 2025-08-14 14:33:35-2025-09-03 14:33:35, threshold 2025-10-25 00:00:00 - 1779 events
Cohort 2025-09-03 14:33:35-2025-09-23 14:33:35, threshold 2025-10-25 00:00:00 - 1004 events


In [56]:
tm = cohort_clusters["regular_retail"][1]
tm[tm["leverage_factor"] > 10]
tm[tm["user_address"] == "0xaC0527DB068990a1190caeC6985aa8F2Ec07b609"][[
    'datetime',
    'event_type',
    'type',
    'debt_after',
    'leverage_factor',
]]


,datetime,event_type,type,debt_after,leverage_factor
1655,2025-08-15 06:56:11,position_open,MarketBorrow,23128.880879,1
1656,2025-08-15 06:56:11,position_open,MarketSupplyCollateral,23128.880879,1
1657,2025-09-28 03:49:59,repay_partial,MarketRepay,8435.189010,1
1658,2025-09-28 03:52:47,collateral_withdraw,MarketWithdrawCollateral,8435.189010,1
1659,2025-09-29 04:26:11,repay_full,MarketRepay,-262.891302,1


In [46]:

# Build cohort clusters
cohort_clusters = {
    "all": [],
    "regular_retail": [],
    "large": [],
    # "risky": [],
}

for curr_df in monthly_dfs:
    risky_users = curr_df[curr_df["ltv_after"] > 0.8]["user_address"].unique() 
    large_users = curr_df[curr_df["debt_after"] > 100_000]["user_address"].unique() 
    small_ltv = curr_df.groupby("user_address")["ltv_after"].max()
    small_ltv = small_ltv[small_ltv<0.6].index
    small_size = curr_df.groupby("user_address")["debt_after"].max()
    small_size = small_size[small_size<50_000].index

    cohort_clusters["all"].append(curr_df)
    # cohort_clusters["risky"].append(
    #     curr_df[
    #         curr_df["user_address"].isin(curr_df[curr_df["debt_after"] > 100]["user_address"].unique()) &
    #         curr_df["user_address"].isin(risky_users)
    #     ]
    # )
    cohort_clusters["large"].append(
        curr_df[curr_df["user_address"].isin(large_users)]
    )
    cohort_clusters["regular_retail"].append(
        curr_df[
            # curr_df["user_address"].isin(small_ltv) &
            curr_df["user_address"].isin(small_size)
        ]
    )

    print(f'Large {len(cohort_clusters["large"][-1])}, retail {len(cohort_clusters["regular_retail"][-1])}')


Large 1585, retail 130
Large 1362, retail 298
Large 771, retail 191


In [68]:
market_df

,timestamp,datetime,total_supply,total_borrow,utilization,borrow_rate,supply_rate,volatility_1h,drawdown_1h,volatility_6h,drawdown_6h,collateral_price,loan_asset_price,avg_health_factor,borrow_rate_rolling,supply_rate_rolling,asset_price
0,1753452000,2025-07-25 14:00:00,0.000000,0.000000,0.0,0.000000,0.000000,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.980210
1,1753455600,2025-07-25 15:00:00,1.000000,1.000000,1.0,0.159854,0.159854,0,0,0.001014,0.000000,0.980210,0.999808,3.751500,0.079927,0.079927,0.980210
2,1753459200,2025-07-25 16:00:00,1.000000,1.000000,1.0,0.159854,0.159854,0,0,0.001014,0.000000,0.980210,0.999808,3.751500,0.106569,0.106569,0.981426
3,1753462800,2025-07-25 17:00:00,1.000000,1.000000,1.0,0.159854,0.159854,0,0,0.001014,0.000000,0.980210,0.999808,3.751500,0.119890,0.119890,0.981426
4,1753466400,2025-07-25 18:00:00,1.000000,1.000000,1.0,0.159854,0.159854,0,0,0.001014,0.000000,0.980210,0.999808,3.751500,0.127883,0.127883,0.981803
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4672,1770271200,2026-02-05 06:00:00,-642590.851851,73695.579503,0.0,0.500000,0.000000,0,0,0.000191,0.000000,0.999479,0.999806,1.143154,0.500000,0.000000,0.996900
4673,1770274800,2026-02-05 07:00:00,-642590.851851,73695.579503,0.0,0.500000,0.000000,0,0,0.000191,0.000000,0.999479,0.999806,1.143154,0.500000,0.000000,0.998367
4674,1770278400,2026-02-05 08:00:00,-642590.851851,73695.579503,0.0,0.500000,0.000000,0,0,0.000191,0.000000,0.999479,0.999806,1.143154,0.500000,0.000000,0.997869
4675,1770282000,2026-02-05 09:00:00,-642590.851851,73695.579503,0.0,0.500000,0.000000,0,0,0.000191,0.000000,0.999479,0.999806,1.143154,0.500000,0.000000,0.997869


In [72]:

# =========================================================================
# Create hourly data for each cohort
# =========================================================================
cohort_clusters_hourly = {}
for cluster in list(cohort_clusters.keys())[1:]:
    cohort_clusters_hourly[cluster] = []
    for curr_df in cohort_clusters[cluster]:
        thr = str(curr_df["datetime"].max())[:10]
        hourly = create_hourly_user_dataset(
            curr_df,
            market_df,
            n_hours=1,
            threshold_date=str(thr[:10]),
            additional_action_features=["implied_apy"]
        )
        cohort_clusters_hourly[cluster].append(hourly)

# =========================================================================
# Process and train for regular_retail (example)
# =========================================================================

In [123]:
retail_prepared_hourly = []
retail_prepared_hourly_full = []
retail_prepared = []

thresholds = {
    'market_utilization': 0.05,
    'borrow_rate': 0.03,
    'implied_apy': 0.01,
}

for curr_df in tqdm(cohort_clusters_hourly["regular_retail"], desc="Processing retail cohorts"):
    new = add_time_difference_features(curr_df, [6, 24])
    print(f"Before removing {len(new)}",end=' ')
    retail_prepared_hourly_full.append(new)
    new = remove_similar_hours(new, thresholds, min_events_cnt=2)
    print(f"After removing {len(new)}, {len(new[new['action'] != 'none'])}")
    retail_prepared_hourly.append(new)
    new = create_model_dataset(new, target_horizon_hours=4, leverage_factor_min=0)
    retail_prepared.append(new)

large_prepared_hourly = []
large_prepared_hourly_full = []
large_prepared = []

thresholds = {
    'market_utilization': 0.05,
    'borrow_rate': 0.03,
    'implied_apy': 0.01,
}

for curr_df in tqdm(cohort_clusters_hourly["large"], desc="Processing large cohorts"):
    new = add_time_difference_features(curr_df, [6, 24])
    print(f"Before removing {len(new)}",end=' ')
    large_prepared_hourly_full.append(new)
    new = remove_similar_hours(new, thresholds, min_events_cnt=2)
    print(f"After removing {len(new)}, {len(new[new['action'] != 'none'])}")
    large_prepared_hourly.append(new)
    new = create_model_dataset(new, target_horizon_hours=4, leverage_factor_min=0)
    large_prepared.append(new)


Processing retail cohorts:   0%|          | 0/3 [00:00<?, ?it/s]

Before removing 14357 After removing 2152, 56
Before removing 26411 After removing 2827, 111
Before removing 10802 After removing 1371, 73


Processing large cohorts:   0%|          | 0/3 [00:00<?, ?it/s]

Before removing 31331 After removing 4896, 436
Before removing 26594 After removing 3143, 399
Before removing 9314 After removing 1215, 175


In [58]:
# =========================================================================
# Train models and compute elasticity for regular_retail
# =========================================================================
results = []
models_list = []
model_df_list = []
scaler_list = []

interval_month_len = 1
# exp_name = f"pt_markets/base_cbbtc_usdc_{interval_month_len}month_retail"

for i in range(len(retail_prepared) - interval_month_len + 1):
    model_df = pd.concat(retail_prepared[i:i+interval_month_len], ignore_index=True).drop(columns=[
        "action_count_last_6h",
    ])
    model, result, p_values, coefs, metrics = train_logistic_model_statsmodels(
        model_df,
        verbose=False
    )
    row = {
        'interval_start': cohort_clusters_hourly["regular_retail"][i]["datetime"].min(),
        'interval_end': cohort_clusters_hourly["regular_retail"][i]["datetime"].min(),
        'roc_auc': metrics['roc_auc'],
        'pr_auc': metrics['pr_auc'],
    }
    for feat, p in p_values.items():
        row[f'p_{feat}'] = p
    for feat, c in coefs.items():
        row[f'coef_{feat}'] = c
    row['cohort_users'] = list(cohort_clusters_hourly["regular_retail"][i])
    models_list.append(model)
    results.append(row)
    model_df_list.append(model_df)
    scaler_list.append(result)

feature_names_list = ['borrow_rate_vs_open', 'ltv_vs_open', 'collateral_price_vs_open', 'collateral_price_vs_6h', 'volatility_6h', 'drawdown_6h', 'ltv_change_1h', 'distance_to_liq', 'borrow_rate_trend', 'ltv_times_rate', 'debt_change_1h', 'volatility_ltv', 'log_debt', 'log_dist_to_liq', 'hour_of_day', 'day_of_week']
features_of_interest = ['ltv_vs_open', 'borrow_rate_vs_open', 'collateral_price_vs_open',
                        'ltv_change_1h', 'distance_to_liq',
                        'log_debt', 'ltv_times_rate']

for i, (model_df, model, scaler) in enumerate(zip(model_df_list, models_list, scaler_list)):
    elasticities, p_mean = compute_elasticity(model_df, model, scaler, feature_names_list, features_of_interest)
    results[i]['p_mean'] = p_mean
    for feat, el in elasticities.items():
        results[i][f'elasticity_{feat}'] = el

results_df = pd.DataFrame(results).drop(columns=[
    # "p_const",
    # "cohort_users",
])
# results_df.to_csv(f"/Users/yegortrussov/Documents/ml/lending_protocols/experiments_logs/{exp_name}.csv", index=False)



LinAlgError: Singular matrix

In [85]:
results_df

,interval_start,interval_end,roc_auc,pr_auc,p_const,p_borrow_rate_vs_open,p_ltv_vs_open,p_collateral_price_vs_open,p_collateral_price_vs_6h,p_volatility_6h,p_drawdown_6h,p_ltv_change_1h,p_distance_to_liq,p_borrow_rate_trend,p_ltv_times_rate,p_debt_change_1h,p_volatility_ltv,p_log_debt,p_log_dist_to_liq,p_hour_of_day,p_day_of_week,coef_const,coef_borrow_rate_vs_open,coef_ltv_vs_open,coef_collateral_price_vs_open,coef_collateral_price_vs_6h,coef_volatility_6h,coef_drawdown_6h,coef_ltv_change_1h,coef_distance_to_liq,coef_borrow_rate_trend,coef_ltv_times_rate,coef_debt_change_1h,coef_volatility_ltv,coef_log_debt,coef_log_dist_to_liq,coef_hour_of_day,coef_day_of_week,cohort_users,p_mean,elasticity_ltv_vs_open,elasticity_borrow_rate_vs_open,elasticity_collateral_price_vs_open,elasticity_ltv_change_1h,elasticity_distance_to_liq,elasticity_log_debt,elasticity_ltv_times_rate
0,2025-07-31 16:01:47,2025-07-31 16:01:47,0.063869,0.006350,7.715163e-46,0.087043,0.068824,0.003430,0.682337,0.515583,0.628016,0.141465,0.592636,0.954476,0.741703,0.207744,0.590600,0.277777,0.718241,0.611626,0.135004,-4.855930,-0.601144,0.399974,-1.108827,-0.095395,-0.452965,-0.142040,0.317518,2.191236,0.018976,0.175737,-0.224620,0.401608,0.410300,-1.384107,0.106808,-0.324921,"[user_address, timestamp, datetime, collateral...",0.016308,-3.209457e-02,-0.014886,-0.003766,-1.186117e-03,5.707346e-01,2.657175,1.286539e-02
1,2025-08-15 00:51:23,2025-08-15 00:51:23,0.773438,0.031493,1.182119e-36,0.000789,0.339422,0.011189,0.065465,0.542273,0.638002,0.076431,0.049241,0.000195,0.687621,0.100444,0.217788,0.320544,0.037705,0.024357,0.244530,-4.870439,-1.823648,-0.519179,-0.624356,0.455417,-0.321064,0.117989,0.501807,-7.687308,1.712323,0.372874,-0.333649,0.819226,-0.238951,6.806272,-0.417636,0.211058,"[user_address, timestamp, datetime, collateral...",0.028218,2.410980e-02,-0.051576,-0.001940,-4.295161e-03,-1.171056e+00,-1.822973,3.146386e-02
2,2025-09-04 02:21:35,2025-09-04 02:21:35,0.497076,0.063465,9.999669e-01,0.015393,NaN,0.038864,0.348938,0.468822,0.594982,NaN,NaN,0.221991,NaN,0.408328,NaN,0.596065,0.201145,0.469824,0.090684,-5.162754,-2.845856,-1.325776,-1.631239,0.791201,0.404788,0.381220,-1.293800,1.268800,1.304343,-1.214118,-0.394438,-1.167097,-0.260064,-8.523589,-0.209936,-0.672092,"[user_address, timestamp, datetime, collateral...",0.073211,2.015809e+10,-0.051240,-0.002048,1.967191e+10,1.929179e+10,-1.548636,1.614265e+09


In [107]:
retail_prepared_hourly[0].head(2)
df.head(2)
# retail_prepared_hourly[0]["user_address"].value_counts()

,hash,type,timestamp,user_address,assets,assets_usd,liquidated_assets,liquidated_assets_usd,market,datetime,market_address,total_supply_before,total_borrow_before,total_supply_after,total_borrow_after,utilization_before,utilization_after,tx_actions,borrow_rate_before,supply_rate_before,borrow_rate_after,supply_rate_after,collateral_price,loan_asset_price,collateral_before,collateral_value_before,debt_before,supply_before,ltv_before,collateral_after,collateral_value_after,debt_after,supply_after,ltv_after,health_factor_before,health_factor_after,event_type,vault_flg,volatility_6h,drawdown_6h,trend_6h,volatility_24h,drawdown_24h,trend_24h,event_sequence_type,collateral_asset_symbol,loan_asset_symbol,base_apy,implied_apy,underlying_apy
0,0x7e1617ef4fb357e6621ccaaff1ebdd06769bf43afe26...,MarketSupply,1753454015,0xfeed46c11F57B7126a773EeC6ae9cA7aE1C03C9a,1000000,0.99981,0,0.0,eth_PT-USDe-25SEP2025_usdc,2025-07-25 14:33:35,0x7a5d67805cb78fad2596899e0c83719ba89df353b931...,0.0,0.0,1.0,0.0,0.0,0.0,1,0.009991,0.0,0.009991,0.0,0.98021,0.999808,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.999808,0.0,0.0,0.0,loan_position_supply,False,0.001014,0.0,-0.002668,0.000669,0.0,-0.002594,loan_position_supply,PT-USDe-25SEP2025,USDC,0.149514,0.124577,0
1,0x8b9babbfd5c7743bca71deaf70d86204f55f261bdb6f...,MarketSupplyCollateral,1753454027,0xfeed46c11F57B7126a773EeC6ae9cA7aE1C03C9a,4182778368794458624,4.10000,0,0.0,eth_PT-USDe-25SEP2025_usdc,2025-07-25 14:33:47,0x7a5d67805cb78fad2596899e0c83719ba89df353b931...,1.0,0.0,1.0,0.0,0.0,0.0,1,0.009991,0.0,0.009991,0.0,0.98021,0.999808,0.0,0.0,0.0,0.999808,0.0,4.182778,4.1,0.0,0.999808,0.0,0.0,0.0,collateral_add,False,0.001014,0.0,-0.002668,0.000669,0.0,-0.002594,borrow_more_w_collateral,PT-USDe-25SEP2025,USDC,0.149514,0.124577,0


In [185]:
market_df["luquidity"] = market_df["total_supply"] - market_df["total_borrow"]
plot_market_features(
    market_df,
    ["luquidity", "implied_apy"],
    y_ranges={"borrow_rate": [0,0.2], "implied_apy": [0,0.2]}
    # dates_range=("2022-01-01", "2025-09-30")
)

In [175]:
df[df["type"] == "MarketSupply"]["user_address"].value_counts()

user_address
0xBEeFFF209270748ddd194831b3fa287a5386f5bC    2150
0xc582F04d8a82795aa2Ff9c8bb4c1c889fe7b754e    1719
0x8eB67A509616cd6A7c1B3c8C21D48FF57df3d458    1247
0x777791C4d6DC2CE140D00D2828a7C93503c67777    1036
0xd63070114470f685b75B74D60EEc7c1113d33a3D     401
0x0C36327e93F749a7eec04603410dF776150f47DE     368
0xbEEf390D2e65d6E43A67875106d4A48f700F2832     238
0x974c8FBf4fd795F66B85B73ebC988A51F1A040a9     225
0xd41830d88dfD08678b0B886E0122193d54b02Acc     128
0xbeEf96B330ef1Fe7Ebe41eCe0Bd4a41A94Bc03DC      58
0x55555815a5595991C3A0Ff119B59AEF6C8B55555      34
0xF9bdDd4A9b3A45f980e11fDDE96e16364dDBEc49      21
0xDCD35020c5bB97016358578131f012Baa9F53cf3      19
0x0F359FD18BDa75e9c49bC027E7da59a4b01BF32a      18
0x7BF6349aE5D9DE5b44416Eb85baA41Cc5ec3a666      16
0x62fE596d59fB077c2Df736dF212E0AFfb522dC78      13
0xa8c18057c5AaaaFF6000D28A057a6994B2e09BFb       9
0x1E2aAaDcF528b9cC08F43d4fd7db488cE89F5741       9
0x000001ac4e512d670c34feDf6c71cE2F49fb160a       8
0xc2A119EA6De75e4B

In [182]:
df[(df["datetime"] >= "2025-08-28 10:00:00") & (df["datetime"] < "2025-08-29 00:00:00")][[
    'datetime',
    'type',
    'total_borrow_after',
    'total_supply_after',
    'utilization_after',
    'borrow_rate_after',
    'supply_rate_after',
    'user_address',
    # 'liquidity'
]][:100]


,datetime,type,total_borrow_after,total_supply_after,utilization_after,borrow_rate_after,supply_rate_after,user_address
7477,2025-08-28 10:03:11,MarketSupply,2.500430e+08,3.015825e+08,0.829103,0.091759,0.076078,0xBEeFFF209270748ddd194831b3fa287a5386f5bC
7478,2025-08-28 10:04:35,MarketWithdraw,2.500430e+08,3.015057e+08,0.829314,0.091776,0.076112,0xBEeFFF209270748ddd194831b3fa287a5386f5bC
7479,2025-08-28 10:08:47,MarketWithdraw,2.500432e+08,3.015059e+08,0.829314,0.091776,0.076112,0xBEeFFF209270748ddd194831b3fa287a5386f5bC
7480,2025-08-28 10:09:23,MarketSupply,2.500432e+08,3.044700e+08,0.821241,0.091120,0.074832,0xBEeFFF209270748ddd194831b3fa287a5386f5bC
7481,2025-08-28 10:17:47,MarketWithdrawCollateral,2.500436e+08,3.044704e+08,0.821241,0.091120,0.074832,0xDbc652411605f8bB8969136923aa039c79989482
7482,2025-08-28 10:23:11,MarketWithdraw,2.500438e+08,3.043265e+08,0.821630,0.091152,0.074894,0xBEeFFF209270748ddd194831b3fa287a5386f5bC
7483,2025-08-28 10:24:23,MarketSupply,2.500439e+08,3.076474e+08,0.812761,0.090431,0.073500,0x0C36327e93F749a7eec04603410dF776150f47DE
7484,2025-08-28 10:25:47,MarketWithdraw,2.500440e+08,3.013994e+08,0.829610,0.091800,0.076159,0xc582F04d8a82795aa2Ff9c8bb4c1c889fe7b754e
7485,2025-08-28 10:26:11,MarketWithdraw,2.500440e+08,2.980785e+08,0.838853,0.092551,0.077638,0x0C36327e93F749a7eec04603410dF776150f47DE
7486,2025-08-28 10:34:11,MarketSupply,2.500443e+08,2.981164e+08,0.838747,0.092543,0.077621,0xc582F04d8a82795aa2Ff9c8bb4c1c889fe7b754e


In [142]:
large_prepared_hourly_full[2][large_prepared_hourly_full[2]["user_address"] == "0xCF263cEe139763114fAaFC5F52865135412F50Ec"][[
    'datetime',
    "collateral",
    "debt",
    "borrow_rate",
    "implied_apy",
]]
# df[df["user_address"] == "0xCF263cEe139763114fAaFC5F52865135412F50Ec"][[
#     'datetime',
#     'event_type',
#     'type',
#     'debt_after',
#     'ltv_after',
# ]]
# large_prepared_hourly[2].head()

,datetime,collateral,debt,borrow_rate,implied_apy
6209,2025-09-07 19:38:23,70616.278156,54985.370000,0.090256,0.166211
6210,2025-09-07 20:38:23,176262.194376,104972.070000,0.089709,0.166329
6211,2025-09-07 21:38:23,176262.194376,104972.070000,0.089633,0.166299
6212,2025-09-07 22:38:23,176262.194376,104972.070000,0.089634,0.166949
6213,2025-09-07 23:38:23,176262.194376,104972.070000,0.089478,0.167108
...,...,...,...,...,...
6635,2025-09-25 13:38:23,176262.194376,104972.070000,0.067870,0.340891
6636,2025-09-25 14:38:23,176262.194376,104972.070000,0.067455,0.340891
6637,2025-09-25 15:38:23,176262.194376,104972.070000,0.063395,0.340891
6638,2025-09-25 16:38:23,176262.194376,104972.070000,0.069596,0.340891


In [134]:
compute_position_metrics(
    large_prepared_hourly_full[2],
    "0xd3fC2D1f183f509AAAa814541055A11a0358d8E9"
)

{'total_yield_usd': 32807.996567460286,
 'total_interest_usd': 13320.937915585875,
 'net_profit_usd': 19487.05865187441,
 'avg_collateral_usd': 3047369.2434316557,
 'avg_debt_usd': 2628605.746707182,
 'effective_yield_apy': 293.0041698746393,
 'effective_borrow_apr': 137.92040422990434,
 'position_days': 22.124999999999787,
 'initial_collateral_usd': 3047369.243431612,
 'final_collateral_usd': 0.0,
 'initial_debt_usd': 2628605.7467071917,
 'final_debt_usd': -13761.264461337909,
 'open_timestamp': 1757145707,
 'close_timestamp': 1759057307}

In [ ]:
# from visualization_utils import plot_user_metrics
# cohort_index = 2
# df_for_vis = large_prepared_hourly
# for ua in df_for_vis[cohort_index]["user_address"].value_counts().index:
#     plot_user_metrics(
#         df_for_vis[cohort_index],
#         ['debt', 'implied_apy'],
#         ua,
#         dates_range=("2025-01-01", "2025-10-25"),
#     )

In [162]:
plot_user_metrics(
    pd.concat(large_prepared_hourly_full,ignore_index=True),
    ['debt', 'implied_apy'],
    "0x23b53B17e22034e1EdC3C5e04Fbbf504CAe9B6BB",
    dates_range=("2025-01-01", "2025-10-25"),
)


    User 0x23b53B17e22034e1EdC3C5e04Fbbf504CAe9B6BB
    Max debt 2449536.95
    Max LTV 0.8669315209432102
    Position opens 1
    


In [156]:
def analyze_positions(hourly_df):
    # Get last row per user (position close)
    last_rows = hourly_df.groupby('user_address').last().reset_index()
    
    # Position lengths (days)
    first_rows = hourly_df.groupby('user_address').first().reset_index()
    pos_length = (last_rows['timestamp'] - first_rows['timestamp']) / (24 * 3600)
    
    print("\n=== Position Closing Dates ===")
    print(last_rows['datetime'].dt.date.value_counts()[:30].sort_index())
    print("\n=== Position opening Dates ===")
    print(first_rows['datetime'].dt.date.value_counts().head(10))

    # print(last_rows[last_rows['datetime'].astype(str) >= "2025-10-01"]["user_address"].unique())
    print(last_rows[last_rows['datetime'].astype(str) < "2025-09-01"]["user_address"].unique())
    
    # print("\n=== Debt Statistics ===")
    # print(first_rows['debt'].describe())
    
    # print(f"\nTotal users: {len(last_rows)}")
    # print(f"Users with zero final debt: {(last_rows['debt'] == 0).sum()}")
    # print(f"Users with negative final debt (over-repaid): {(last_rows['debt'] < 0).sum()}")
analyze_positions(
    pd.concat(large_prepared_hourly,ignore_index=True)
)


=== Position Closing Dates ===
datetime
2025-08-08     1
2025-08-13     1
2025-08-14     2
2025-08-17     2
2025-08-18     3
2025-08-19     2
2025-08-20     3
2025-08-22     3
2025-08-23     3
2025-08-25     3
2025-08-26     1
2025-08-30     2
2025-09-02     2
2025-09-05     1
2025-09-07     1
2025-09-11     2
2025-09-13     2
2025-09-14     2
2025-09-16     3
2025-09-18     2
2025-09-20     3
2025-09-21     5
2025-09-22     7
2025-09-23     5
2025-09-24     7
2025-09-25    35
2025-09-26     4
2025-09-27     5
2025-09-28     1
2025-10-01     3
Name: count, dtype: int64

=== Position opening Dates ===
datetime
2025-08-13    11
2025-08-16     8
2025-08-19     7
2025-08-11     6
2025-08-10     6
2025-09-09     5
2025-08-12     5
2025-09-10     4
2025-08-22     4
2025-08-24     4
Name: count, dtype: int64
['0x01c2f22754E1C2cc64A18Ba6162E5C7d417ffF7f'
 '0x08E63b00B38eA99d745edF61FB653f8949aEA5e0'
 '0x102407F67415dCc4068370625Ca27F24bB2A03d5'
 '0x16f037a3dDf53dA1b047A926E1833219F0a8E1FC'
 '